In [ ]:
# ROGII Wellbore Geology Prediction - v9 submission notebook
#
# v9 = v8 + neural-ANCC global surface features stacked into the LGB +
# XGB + Ridge ensemble.
#
# konbu17 (LB 11.912) uses LOCAL spatial estimators only: per-well plane
# fit (K=10) and row-level KNN (K=20). Both fail on sparse-neighbor wells,
# producing the catastrophic-tail outliers we see in v8 OOF (max well
# 56 ft, p90 17 ft).
#
# Empirically (rogii/bench/neural_ancc_results.json, full 5-fold OOF over
# 765 wells / 5,040,554 rows): a 4-layer x 256 NeRF-MLP with sinusoidal PE
# (L=8) and multi-output head (all 6 formations) BEATS row-level KNN on
# the catastrophic tail by 4x:
#   wells with TVT RMSE > 60 ft : 46 (KNN) -> 11 (MLP)
#   pooled ANCC RMSE            : 30.74 (KNN) -> 24.10 (MLP multi)
#   max well RMSE               : 300.85 (KNN) -> 165.66 (MLP multi)
# but loses on the typical median (12.30 vs 14.67) because KNN is sharper
# in dense-neighbor regions.
#
# v9's play: feed BOTH KNN and MLP predictions to the LGB stack and let
# the GBM learn the gate from knn_row_dist + neighbor stds. EWM(span=4)
# post-smoothing per well retained.

import os
import sys
import base64
import json
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
)
logger = logging.getLogger("rogii.v9")

# ---------------------------------------------------------------------------
# 1) Write the modules to /kaggle/working and import them.
# ---------------------------------------------------------------------------
SRC_DIR = "/kaggle/working/rogii_src"
os.makedirs(SRC_DIR, exist_ok=True)

_MODULES = {
    "feature_builder.py": "...",
    "neural_ancc.py": "IiIiTmV1cmFsIEFOQ0MoWCwgWSkgc3VyZmFjZSBtb2RlbC4KCkh5cG90aGVzaXMgKGZyb20gU1RSQVRFR1lfUkVTRVQpOiBrb25idTE3J3MgcGVyLXdlbGwgcGxhbmUgZml0IGFuZCByb3ctbGV2ZWwKS05OIGFyZSAqbG9jYWwqIOKAlCB0aGV5IGZhaWwgaW4gc3BhdGlhbCByZWdpb25zIHdpdGggc3BhcnNlIHRyYWluaW5nIG5laWdoYm9ycywKcHJvZHVjaW5nIHRoZSBjYXRhc3Ryb3BoaWMgd2VsbC1STVNFIG91dGxpZXJzIHdlIHNlZSBpbiB2OCBPT0YgKG1heCA1NiBmdCkuCkEgc21hbGwgTUxQIHdpdGggc2ludXNvaWRhbCBwb3NpdGlvbmFsIGVuY29kaW5nIChOZVJGLXN0eWxlKSBvbiAoWCwgWSkgbGVhcm5zCmEgKmdsb2JhbCBzbW9vdGggc3VyZmFjZSogdGhhdCBzaG91bGQgZ2VuZXJhbGl6ZSBiZXR0ZXIgaW4gdGhvc2Ugc3BhcnNlIHJlZ2lvbnMKd2hpbGUgc3RpbGwgbWF0Y2hpbmcgbG9jYWwgZmVhdHVyZXMgdmlhIHRoZSBoaWdoLWZyZXF1ZW5jeSBQRSBiYW5kcy4KClRoZSBsb2FkLWJlYXJpbmcgaWRlbnRpdHk6CiAgICBUVlQgPSAtWiArIEFOQ0MgKyBiX3dlbGwgICAoaW50cmEtd2VsbCBzaWdtYSAwLjAwNjUgZnQsIGV4YWN0KQoKU28gcHJlZGljdGluZyBBTkNDKFgsIFkpIGF0IGEgaGVsZC1vdXQgd2VsbCdzIChYLCBZKSwgdGhlbiBwbHVnZ2luZyBpbnRvIHRoZQpjbG9zZWQtZm9ybSBUVlQgZm9ybXVsYSB3aXRoIHRoZSB3ZWxsJ3MgbWVkaWFuIGJfd2VsbCBmcm9tIGl0cyB2aXNpYmxlIHByZWZpeCwKaXMgc3VmZmljaWVudCB0byByZWNvdmVyIFRWVCB3aXRoIHRoZSBzYW1lIGZpZGVsaXR5IGFzIEFOQ0MuCgpEZXNpZ24gKHBlciB0aGUgYnJpZWYsIG5vIHR1bmluZyBzYWdhKToKICAxLiAoWCwgWSkgbm9ybWFsaXplZCB0byBbLTEsIDFdIG92ZXIgdGhlIHRyYWluIGV4dGVudC4KICAyLiBTaW51c29pZGFsIHBvc2l0aW9uYWwgZW5jb2Rpbmc6IGdhbW1hKHApID0gW3NpbigyXmsgKiBwaSAqIHApLCBjb3MoLi4uKV0KICAgICBmb3IgayA9IDAuLkwtMSwgYXBwbGllZCB0byBYIGFuZCBZIHNlcGFyYXRlbHkuIE91dHB1dCBkaW0gPSA0ICogTCBwZXIgY29vcmQuCiAgICAgUGx1cyB0aGUgcmF3IChYLCBZKSBmZWF0dXJlIGNvbmNhdGVuYXRlZC4KICAzLiBNTFA6IDQgbGF5ZXJzIHggMjU2LCBTaUxVLCBOZVJGLXN0eWxlIHNraXAgZnJvbSBpbnB1dCB0byBsYXllciAyLgogIDQuIEFkYW0sIGxyPTFlLTMsIGNvc2luZSBkZWNheSwgYmF0Y2ggNDA5NiwgNTAwayByb3dzIC8gZXBvY2gsIE1QUyBiYWNrZW5kLgogIDUuIFNxdWFyZWQgbG9zcyBvbiBBTkNDLiBNdWx0aS1vdXRwdXQgdmFyaWFudCBwcmVkaWN0cyBhbGwgNiBmb3JtYXRpb24gdG9wcy4KCkFsbCB0cmFpbmluZyBpcyBwZXItZm9sZCAodHJhaW4gZm9sZCByb3dzIG9ubHkpLiBJbmZlcmVuY2U6IG1vZGVsLnByZWRpY3RfeHkKb24gdGhlIGhlbGQtb3V0IChYLCBZKSAtPiBBTkNDIC0+ICgtWiArIEFOQ0MgKyBiX3ByZWZpeF9tZWRpYW4pIC0+IFRWVC4KClRoaXMgbW9kdWxlIGlzIHNlbGYtY29udGFpbmVkIOKAlCBubyBwYW5kYXMsIHBvbGFycyArIHRvcmNoIG9ubHkuIFRoZSA1TSAoWCwgWSwKZm9ybWF0aW9uKSB0dXBsZXMgZml0IGNvbWZvcnRhYmx5IGluIDMyR0IuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUsIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBvbGFycyBhcyBwbAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpGT1JNQVRJT05TOiB0dXBsZVtzdHIsIC4uLl0gPSAoIkFOQ0MiLCAiQVNUTlUiLCAiQVNUTkwiLCAiRUdGRFUiLCAiRUdGREwiLCAiQlVEQSIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBEYXRhIGxvYWRpbmcKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBsb2FkX3RyYWluX3Jvd3MoCiAgICB0cmFpbl9kaXI6IFBhdGgsCiAgICBmb3JtYXRpb25zOiBTZXF1ZW5jZVtzdHJdID0gRk9STUFUSU9OUywKICAgIHBhdGhzOiBJdGVyYWJsZVtQYXRoXSB8IE5vbmUgPSBOb25lLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiTG9hZCBhbGwgKFgsIFksIGZvcm1hdGlvbnNbLCB3ZWxsXSkgcm93cyBmcm9tIHRyYWluaW5nIENTVnMuCgogICAgUmV0dXJucwogICAgLS0tLS0tLQogICAgeHkgOiAoTiwgMikgZmxvYXQ2NAogICAgdGFyZ2V0cyA6IChOLCBGKSBmbG9hdDMyICAgRiA9IGxlbihmb3JtYXRpb25zKQogICAgd2lkcyA6IChOLCkgb2JqZWN0IHN0ciAgICB3ZWxsIElEIHBlciByb3cKICAgICIiIgogICAgaWYgcGF0aHMgaXMgTm9uZToKICAgICAgICBwYXRocyA9IHNvcnRlZChQYXRoKHRyYWluX2RpcikuZ2xvYigiKl9faG9yaXpvbnRhbF93ZWxsLmNzdiIpKQogICAgY29scyA9IFsiWCIsICJZIiwgKmZvcm1hdGlvbnNdCiAgICB4eV9ibG9ja3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgZl9ibG9ja3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgd2lkX2Jsb2NrczogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICBza2lwcGVkID0gMAogICAgZm9yIHAgaW4gcGF0aHM6CiAgICAgICAgd2lkID0gcC5zdGVtLnJlcGxhY2UoIl9faG9yaXpvbnRhbF93ZWxsIiwgIiIpCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIEZvcmNlIEFOQ0MgZmxvYXQgKHNvbWUgd2VsbHMgc3RvcmUgaXQgYXMgVXRmOCB3aXRoIGFsbC1udWxsKTsKICAgICAgICAgICAgIyBwb2xhcnMgcmVhZF9jc3Ygc2NoZW1hX292ZXJyaWRlcyBoYW5kbGVzIGVpdGhlci4KICAgICAgICAgICAgZGYgPSBwbC5yZWFkX2NzdihwLCBjb2x1bW5zPWNvbHMsIGluZmVyX3NjaGVtYV9sZW5ndGg9MTBfMDAwKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgICMgQ29lcmNlIGFsbCBmb3JtYXRpb25zIHRvIEZsb2F0NjQgaWYgdGhleSBjYW1lIGJhY2sgYXMgVXRmOC4KICAgICAgICBmb3IgYyBpbiBmb3JtYXRpb25zOgogICAgICAgICAgICBpZiBkZltjXS5kdHlwZSA9PSBwbC5VdGY4OgogICAgICAgICAgICAgICAgZGYgPSBkZi53aXRoX2NvbHVtbnMocGwuY29sKGMpLmNhc3QocGwuRmxvYXQ2NCwgc3RyaWN0PUZhbHNlKSkKICAgICAgICBkZiA9IGRmLmRyb3BfbnVsbHMoc3Vic2V0PWNvbHMpCiAgICAgICAgaWYgbGVuKGRmKSA9PSAwOgogICAgICAgICAgICBza2lwcGVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICB4eV9ibG9ja3MuYXBwZW5kKGRmLnNlbGVjdChbIlgiLCAiWSJdKS50b19udW1weSgpLmFzdHlwZShucC5mbG9hdDY0KSkKICAgICAgICBmX2Jsb2Nrcy5hcHBlbmQoZGYuc2VsZWN0KGxpc3QoZm9ybWF0aW9ucykpLnRvX251bXB5KCkuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAgIHdpZF9ibG9ja3MuYXBwZW5kKG5wLmZ1bGwobGVuKGRmKSwgd2lkLCBkdHlwZT1vYmplY3QpKQogICAgaWYgbm90IHh5X2Jsb2NrczoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJObyB0cmFpbmluZyByb3dzIGxvYWRlZCBmcm9tIHt0cmFpbl9kaXJ9IikKICAgIHh5ID0gbnAuY29uY2F0ZW5hdGUoeHlfYmxvY2tzKQogICAgdGFyZ2V0cyA9IG5wLmNvbmNhdGVuYXRlKGZfYmxvY2tzKQogICAgd2lkcyA9IG5wLmNvbmNhdGVuYXRlKHdpZF9ibG9ja3MpCiAgICByZXR1cm4geHksIHRhcmdldHMsIHdpZHMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE5ldXJhbCBtb2RlbAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgUG9zaXRpb25hbEVuY29kaW5nKG5uLk1vZHVsZSk6CiAgICAiIiJOZVJGLXN0eWxlIHNpbnVzb2lkYWwgcG9zaXRpb25hbCBlbmNvZGluZyBvbiBlYWNoIGNvb3JkaW5hdGUuCgogICAgcCBhc3N1bWVkIG5vcm1hbGl6ZWQgdG8gcm91Z2hseSBbLTEsIDFdLiBPdXRwdXQgZGltID0gNCpMIChjb3Mvc2luIHggMiBjb29yZHMpLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9mcmVxczogaW50KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm51bV9mcmVxcyA9IG51bV9mcmVxcwogICAgICAgICMgMl5rICogcGkgZm9yIGsgPSAwIC4uIEwtMQogICAgICAgIGZyZXFzID0gKDIuMCAqKiB0b3JjaC5hcmFuZ2UobnVtX2ZyZXFzKSkgKiBtYXRoLnBpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImZyZXFzIiwgZnJlcXMudG8odG9yY2guZmxvYXQzMikpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeHk6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICMgeHk6IChCLCAyKQogICAgICAgIGlmIHNlbGYubnVtX2ZyZXFzID09IDA6CiAgICAgICAgICAgIHJldHVybiB4eQogICAgICAgIHNjYWxlZCA9IHh5LnVuc3F1ZWV6ZSgtMSkgKiBzZWxmLmZyZXFzICAgIyAoQiwgMiwgTCkKICAgICAgICBzaW4gPSB0b3JjaC5zaW4oc2NhbGVkKQogICAgICAgIGNvcyA9IHRvcmNoLmNvcyhzY2FsZWQpCiAgICAgICAgIyBpbnRlcmxlYXZlIHRvIChCLCA0ICogTCkKICAgICAgICBlbmNvZGVkID0gdG9yY2guY2F0KFtzaW4sIGNvc10sIGRpbT0tMSkuZmxhdHRlbihzdGFydF9kaW09MSkKICAgICAgICByZXR1cm4gdG9yY2guY2F0KFt4eSwgZW5jb2RlZF0sIGRpbT0tMSkgICMgcmF3ICsgUEUKCgpjbGFzcyBOZXJmTUxQKG5uLk1vZHVsZSk6CiAgICAiIiI0IGhpZGRlbiBsYXllcnMgeCAyNTYsIFNpTFUsIHdpdGggc2tpcCBmcm9tIGlucHV0IHRvIGxheWVyIDIuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2RpbTogaW50LCBoaWRkZW46IGludCwgb3V0X2RpbTogaW50KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmZjMSA9IG5uLkxpbmVhcihpbl9kaW0sIGhpZGRlbikKICAgICAgICBzZWxmLmZjMiA9IG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbikKICAgICAgICBzZWxmLmZjMyA9IG5uLkxpbmVhcihoaWRkZW4gKyBpbl9kaW0sIGhpZGRlbikgICAjIHNraXAKICAgICAgICBzZWxmLmZjNCA9IG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbikKICAgICAgICBzZWxmLmhlYWQgPSBubi5MaW5lYXIoaGlkZGVuLCBvdXRfZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGggPSBGLnNpbHUoc2VsZi5mYzEoeCkpCiAgICAgICAgaCA9IEYuc2lsdShzZWxmLmZjMihoKSkKICAgICAgICBoID0gRi5zaWx1KHNlbGYuZmMzKHRvcmNoLmNhdChbaCwgeF0sIGRpbT0tMSkpKQogICAgICAgIGggPSBGLnNpbHUoc2VsZi5mYzQoaCkpCiAgICAgICAgcmV0dXJuIHNlbGYuaGVhZChoKQoKCkBkYXRhY2xhc3MKY2xhc3MgVHJhaW5Db25maWc6CiAgICBudW1fZnJlcXM6IGludCA9IDgKICAgIGhpZGRlbjogaW50ID0gMjU2CiAgICBvdXRfZGltOiBpbnQgPSAxCiAgICByb3dzX3Blcl9lcG9jaDogaW50ID0gNTAwXzAwMAogICAgYmF0Y2hfc2l6ZTogaW50ID0gNDA5NgogICAgZXBvY2hzOiBpbnQgPSAxMgogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMAogICAgc2VlZDogaW50ID0gNDIKICAgIGRldmljZTogc3RyID0gIm1wcyIgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgIHZhbF9mcmFjOiBmbG9hdCA9IDAuMCAgIyBubyBpbnRlcm5hbCB2YWw6IGV4dGVybmFsIEdyb3VwS0ZvbGQgb3ducyB2YWwuCgoKY2xhc3MgQW5jY05ldDoKICAgICIiIldyYXBzIHRoZSBtb2RlbCArIHRyYWluIGV4dGVudCBub3JtYWxpemVyICsgdHJhaW4gbG9vcC4KCiAgICBvdXRfZGltPT0xIC0+IEFOQ0Mgb25seS4gb3V0X2RpbT09bGVuKEZPUk1BVElPTlMpIC0+IGFsbC1mb3JtYXRpb25zIGhlYWQuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2ZnOiBUcmFpbkNvbmZpZyk6CiAgICAgICAgc2VsZi5jZmcgPSBjZmcKICAgICAgICB0b3JjaC5tYW51YWxfc2VlZChjZmcuc2VlZCkKICAgICAgICBzZWxmLnBlID0gUG9zaXRpb25hbEVuY29kaW5nKGNmZy5udW1fZnJlcXMpCiAgICAgICAgaW5fZGltID0gMiArICg0ICogY2ZnLm51bV9mcmVxcyBpZiBjZmcubnVtX2ZyZXFzID4gMCBlbHNlIDApCiAgICAgICAgc2VsZi5tbHAgPSBOZXJmTUxQKGluX2RpbSwgY2ZnLmhpZGRlbiwgY2ZnLm91dF9kaW0pCiAgICAgICAgc2VsZi5kZXZpY2UgPSB0b3JjaC5kZXZpY2UoY2ZnLmRldmljZSkKICAgICAgICBzZWxmLnBlLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYubWxwLnRvKHNlbGYuZGV2aWNlKQogICAgICAgICMgdHJhaW4tZXh0ZW50IG5vcm1hbGl6ZXIgKHNldCBpbiBmaXQpCiAgICAgICAgc2VsZi54X21pZCA9IDAuMAogICAgICAgIHNlbGYueF9zY2wgPSAxLjAKICAgICAgICBzZWxmLnlfbWlkID0gMC4wCiAgICAgICAgc2VsZi55X3NjbCA9IDEuMAogICAgICAgICMgdGFyZ2V0IG5vcm1hbGl6ZXIgKG1lYW4gLyBzdGQgcGVyIG91dHB1dCBkaW0sIHNldCBpbiBmaXQpCiAgICAgICAgc2VsZi50X21lYW4gPSBucC56ZXJvcyhjZmcub3V0X2RpbSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBzZWxmLnRfc3RkID0gbnAub25lcyhjZmcub3V0X2RpbSwgZHR5cGU9bnAuZmxvYXQzMikKCiAgICAjIC0tIG5vcm1hbGl6ZXJzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIF9maXRfbm9ybShzZWxmLCB4eTogbnAubmRhcnJheSwgdGFyZ2V0czogbnAubmRhcnJheSkgLT4gTm9uZToKICAgICAgICB4X21pbiwgeF9tYXggPSBmbG9hdCh4eVs6LCAwXS5taW4oKSksIGZsb2F0KHh5WzosIDBdLm1heCgpKQogICAgICAgIHlfbWluLCB5X21heCA9IGZsb2F0KHh5WzosIDFdLm1pbigpKSwgZmxvYXQoeHlbOiwgMV0ubWF4KCkpCiAgICAgICAgc2VsZi54X21pZCA9IDAuNSAqICh4X21pbiArIHhfbWF4KQogICAgICAgIHNlbGYueF9zY2wgPSBtYXgoMC41ICogKHhfbWF4IC0geF9taW4pLCAxLjApCiAgICAgICAgc2VsZi55X21pZCA9IDAuNSAqICh5X21pbiArIHlfbWF4KQogICAgICAgIHNlbGYueV9zY2wgPSBtYXgoMC41ICogKHlfbWF4IC0geV9taW4pLCAxLjApCiAgICAgICAgc2VsZi50X21lYW4gPSB0YXJnZXRzLm1lYW4oYXhpcz0wKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAjIEF2b2lkIGRpdi1ieS16ZXJvIG9uIGRlZ2VuZXJhdGUgY2FzZXMuCiAgICAgICAgc2VsZi50X3N0ZCA9IG5wLm1heGltdW0odGFyZ2V0cy5zdGQoYXhpcz0wKSwgMS4wKS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICBkZWYgX25vcm1feHkoc2VsZiwgeHk6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgb3V0ID0gbnAuZW1wdHlfbGlrZSh4eSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBvdXRbOiwgMF0gPSAoeHlbOiwgMF0gLSBzZWxmLnhfbWlkKSAvIHNlbGYueF9zY2wKICAgICAgICBvdXRbOiwgMV0gPSAoeHlbOiwgMV0gLSBzZWxmLnlfbWlkKSAvIHNlbGYueV9zY2wKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLSB0cmFpbmluZyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgogICAgZGVmIGZpdChzZWxmLCB4eV90cmFpbjogbnAubmRhcnJheSwgdF90cmFpbjogbnAubmRhcnJheSwgKiwgdmVyYm9zZTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OgogICAgICAgIGNmZyA9IHNlbGYuY2ZnCiAgICAgICAgaWYgdF90cmFpbi5uZGltID09IDE6CiAgICAgICAgICAgIHRfdHJhaW4gPSB0X3RyYWluLnJlc2hhcGUoLTEsIDEpCiAgICAgICAgYXNzZXJ0IHRfdHJhaW4uc2hhcGVbMV0gPT0gY2ZnLm91dF9kaW0sICh0X3RyYWluLnNoYXBlLCBjZmcub3V0X2RpbSkKICAgICAgICBzZWxmLl9maXRfbm9ybSh4eV90cmFpbiwgdF90cmFpbikKICAgICAgICB4eV9uID0gc2VsZi5fbm9ybV94eSh4eV90cmFpbikKICAgICAgICB0X24gPSAoKHRfdHJhaW4gLSBzZWxmLnRfbWVhbikgLyBzZWxmLnRfc3RkKS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICAgICAgZGV2aWNlID0gc2VsZi5kZXZpY2UKICAgICAgICB4eV90ID0gdG9yY2guZnJvbV9udW1weSh4eV9uKS50byhkZXZpY2UpCiAgICAgICAgdF90ID0gdG9yY2guZnJvbV9udW1weSh0X24pLnRvKGRldmljZSkKICAgICAgICBOID0geHlfdC5zaGFwZVswXQoKICAgICAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtKHNlbGYubWxwLnBhcmFtZXRlcnMoKSwgbHI9Y2ZnLmxyLCB3ZWlnaHRfZGVjYXk9Y2ZnLndlaWdodF9kZWNheSkKICAgICAgICAjIENvc2luZSBkZWNheSBvdmVyIHRvdGFsIGl0ZXJhdGlvbnMgKGFjcm9zcyBhbGwgZXBvY2hzKS4KICAgICAgICByb3dzX3Blcl9lcG9jaCA9IG1pbihjZmcucm93c19wZXJfZXBvY2gsIE4pCiAgICAgICAgc3RlcHNfcGVyX2Vwb2NoID0gbWF4KHJvd3NfcGVyX2Vwb2NoIC8vIGNmZy5iYXRjaF9zaXplLCAxKQogICAgICAgIHRvdGFsX3N0ZXBzID0gY2ZnLmVwb2NocyAqIHN0ZXBzX3Blcl9lcG9jaAogICAgICAgIHNjaGVkID0gdG9yY2gub3B0aW0ubHJfc2NoZWR1bGVyLkNvc2luZUFubmVhbGluZ0xSKG9wdCwgVF9tYXg9dG90YWxfc3RlcHMpCgogICAgICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhjZmcuc2VlZCkKICAgICAgICBlcG9jaF9sb3NzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgdF9zdGFydCA9IHRpbWUucGVyZl9jb3VudGVyKCkKCiAgICAgICAgc2VsZi5tbHAudHJhaW4oKQogICAgICAgIGZvciBlcCBpbiByYW5nZShjZmcuZXBvY2hzKToKICAgICAgICAgICAgIyBTYW1wbGUgcm93c19wZXJfZXBvY2ggcmFuZG9tIHJvdyBpbmRpY2VzIGZvciB0aGlzIGVwb2NoLgogICAgICAgICAgICBzZWwgPSB0b3JjaC5mcm9tX251bXB5KAogICAgICAgICAgICAgICAgcm5nLmNob2ljZShOLCByb3dzX3Blcl9lcG9jaCwgcmVwbGFjZT1GYWxzZSkuYXN0eXBlKG5wLmludDY0KQogICAgICAgICAgICApLnRvKGRldmljZSkKICAgICAgICAgICAgeHlfZXAgPSB4eV90W3NlbF0KICAgICAgICAgICAgdF9lcCA9IHRfdFtzZWxdCiAgICAgICAgICAgICMgU2h1ZmZsZSB3aXRoaW4gZXBvY2ggaXMgaW1wbGljaXQgYnkgc2FtcGxpbmcuCiAgICAgICAgICAgIG5fbG9zcyA9IDAuMAogICAgICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCByb3dzX3Blcl9lcG9jaCwgY2ZnLmJhdGNoX3NpemUpOgogICAgICAgICAgICAgICAgeGIgPSB4eV9lcFtzOnMgKyBjZmcuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgICAgIHliID0gdF9lcFtzOnMgKyBjZmcuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgICAgIGZlYXRzID0gc2VsZi5wZSh4YikKICAgICAgICAgICAgICAgIHByZWQgPSBzZWxmLm1scChmZWF0cykKICAgICAgICAgICAgICAgIGxvc3MgPSBGLm1zZV9sb3NzKHByZWQsIHliKQogICAgICAgICAgICAgICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICBvcHQuc3RlcCgpCiAgICAgICAgICAgICAgICBzY2hlZC5zdGVwKCkKICAgICAgICAgICAgICAgIG5fbG9zcyArPSBsb3NzLml0ZW0oKSAqIHhiLnNoYXBlWzBdCiAgICAgICAgICAgIGF2ZyA9IG5fbG9zcyAvIHJvd3NfcGVyX2Vwb2NoCiAgICAgICAgICAgIGVwb2NoX2xvc3MuYXBwZW5kKGF2ZykKICAgICAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgICAgIHByaW50KGYiICBlcCB7ZXA6MDJkfSAgbG9zcyhub3JtKT17YXZnOi41Zn0gIGxyPXtvcHQucGFyYW1fZ3JvdXBzWzBdWydsciddOi4yZX0iLCBmbHVzaD1UcnVlKQogICAgICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdF9zdGFydAogICAgICAgIHJldHVybiB7ImVwb2NoX2xvc3MiOiBlcG9jaF9sb3NzLCAiZml0X3RpbWVfcyI6IGVsYXBzZWR9CgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHByZWRpY3Qoc2VsZiwgeHk6IG5wLm5kYXJyYXksICosIGJhdGNoX3NpemU6IGludCA9IDIwMF8wMDApIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiUHJlZGljdCB0YXJnZXRzIGF0IHh5LiBSZXR1cm5zIChOLCBvdXRfZGltKSBmbG9hdDMyIGluIG9yaWdpbmFsIHVuaXRzLiIiIgogICAgICAgIHNlbGYubWxwLmV2YWwoKQogICAgICAgIHh5X24gPSBzZWxmLl9ub3JtX3h5KHh5KQogICAgICAgIHh5X3QgPSB0b3JjaC5mcm9tX251bXB5KHh5X24pLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIG91dHM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgICAgIGZvciBzIGluIHJhbmdlKDAsIHh5X3Quc2hhcGVbMF0sIGJhdGNoX3NpemUpOgogICAgICAgICAgICBmZWF0cyA9IHNlbGYucGUoeHlfdFtzOnMgKyBiYXRjaF9zaXplXSkKICAgICAgICAgICAgcHJlZCA9IHNlbGYubWxwKGZlYXRzKS5jcHUoKS5udW1weSgpCiAgICAgICAgICAgIG91dHMuYXBwZW5kKHByZWQpCiAgICAgICAgb3V0ID0gbnAuY29uY2F0ZW5hdGUob3V0cywgYXhpcz0wKQogICAgICAgIG91dCA9IG91dCAqIHNlbGYudF9zdGRbTm9uZSwgOl0gKyBzZWxmLnRfbWVhbltOb25lLCA6XQogICAgICAgIHJldHVybiBvdXQuYXN0eXBlKG5wLmZsb2F0MzIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDbG9zZWQtZm9ybSBiX3dlbGwgZnJvbSBwcmVmaXgKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBmaXRfYl9wcmVmaXhfbWVkaWFuKAogICAgcHJlZml4X3R2dF9pbnB1dDogbnAubmRhcnJheSwgcHJlZml4X3o6IG5wLm5kYXJyYXksIHByZWZpeF9hbmNjX3ByZWQ6IG5wLm5kYXJyYXkKKSAtPiBmbG9hdDoKICAgICIiIk1lZGlhbiBwZXItcm93IGIgPSBUVlRfaW5wdXQgKyBaIC0gQU5DQ19wcmVkIG92ZXIgdGhlIHZpc2libGUgcHJlZml4LiIiIgogICAgcmVzID0gcHJlZml4X3R2dF9pbnB1dCArIHByZWZpeF96IC0gcHJlZml4X2FuY2NfcHJlZAogICAgcmVzID0gcmVzW25wLmlzZmluaXRlKHJlcyldCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKHJlcykpIGlmIHJlcy5zaXplIGVsc2UgMC4wCg==",
}
for _name, _payload in _MODULES.items():
    with open(os.path.join(SRC_DIR, _name), "wb") as _f:
        _f.write(base64.b64decode(_payload))

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ---------------------------------------------------------------------------
# 2) Discover the competition data root under /kaggle/input/.
# ---------------------------------------------------------------------------
INPUT_ROOT = "/kaggle/input"
DATA_ROOT = None
if os.path.isdir(INPUT_ROOT):
    for root, dirs, _files in os.walk(INPUT_ROOT):
        depth = root.replace(INPUT_ROOT, "").count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        if "test" in dirs and "train" in dirs:
            DATA_ROOT = root
            logger.info("Found competition data at %s (depth %d)", DATA_ROOT, depth)
            break
if DATA_ROOT is None:
    raise SystemExit("FATAL: could not locate competition test/+train/ directories")

TRAIN_DIR = Path(DATA_ROOT) / "train"
TEST_DIR = Path(DATA_ROOT) / "test"
n_train = sum(1 for f in TRAIN_DIR.iterdir() if f.name.endswith("__horizontal_well.csv"))
n_test = sum(1 for f in TEST_DIR.iterdir() if f.name.endswith("__horizontal_well.csv"))
logger.info("train wells: %d  test wells: %d", n_train, n_test)

# ---------------------------------------------------------------------------
# 3) Imports
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import lightgbm as lgb
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as _xgb_exc:
    logger.warning("XGBoost unavailable: %s", _xgb_exc)
    HAS_XGB = False

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

from feature_builder import (
    FORMATIONS,
    FormationPlaneKNN,
    RowKNN,
    MLPAnccImputer,
    build_dataset,
)

PRIMARY_FORMATION = "ANCC"
N_SPLITS = 5
SPLIT_SEED = 42
LGB_SEEDS = [42, 7, 123]
ENABLE_BEAM = True
EWM_SPAN = 4.0
USE_GPU = True

# Neural-ANCC config (matches the OOF-validated MLP+PE-L8 multi-output)
MLP_NUM_FREQS = 8
MLP_HIDDEN = 256
MLP_EPOCHS = 12
MLP_ROWS_PER_EPOCH = 500_000
MLP_SEED = 42

OUTPUT = Path("/kaggle/working/submission.csv")
OOF_OUT = Path("/kaggle/working/oof.csv")

train_paths = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
test_paths = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
logger.info("train paths=%d  test paths=%d", len(train_paths), len(test_paths))

logger.info("Building plane-fit formation imputer ...")
formation_imputer = FormationPlaneKNN.fit(train_paths, formations=FORMATIONS)
logger.info("  %d wells", len(formation_imputer.df))

logger.info("Building row-level KNN imputer ...")
row_imputer = RowKNN.fit(train_paths, formations=FORMATIONS)
logger.info("  %d rows", len(row_imputer.targets))

# ---------------------------------------------------------------------------
# 4) Train the neural ANCC field once on all train rows.
#    The test wells are NOT in train at submission time, so no fold logic.
# ---------------------------------------------------------------------------
logger.info("Training neural-ANCC field (MLP+PE L=%d, hidden=%d, %d epochs) ...",
            MLP_NUM_FREQS, MLP_HIDDEN, MLP_EPOCHS)
mlp_imputer = MLPAnccImputer.fit(
    train_paths, formations=FORMATIONS,
    num_freqs=MLP_NUM_FREQS, hidden=MLP_HIDDEN,
    epochs=MLP_EPOCHS, rows_per_epoch=MLP_ROWS_PER_EPOCH,
    seed=MLP_SEED, verbose=False,
)
logger.info("  MLP fit done (out_dim=%d)", mlp_imputer.net.cfg.out_dim)

logger.info("Building train features ...")
train_df = build_dataset(
    train_paths, formation_imputer, row_imputer,
    is_train=True, mlp_imputer=mlp_imputer,
    primary_formation=PRIMARY_FORMATION,
    formations=FORMATIONS, enable_beam=ENABLE_BEAM, label="train",
)
logger.info("  train shape: %s", train_df.shape)

logger.info("Building test features ...")
test_df = build_dataset(
    test_paths, formation_imputer, row_imputer,
    is_train=False, mlp_imputer=mlp_imputer,
    primary_formation=PRIMARY_FORMATION,
    formations=FORMATIONS, enable_beam=ENABLE_BEAM, label="test",
)
logger.info("  test shape: %s", test_df.shape)

if train_df.empty:
    raise SystemExit("FATAL: empty train feature matrix")
if test_df.empty:
    raise SystemExit("FATAL: empty test feature matrix")

feature_cols = [c for c in train_df.columns if c not in {"well", "prediction_id", "target"}]
logger.info("  #features: %d", len(feature_cols))

# ---------------------------------------------------------------------------
# 5) GroupKFold splits
# ---------------------------------------------------------------------------
gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SPLIT_SEED)
splits = list(gkf.split(train_df, train_df["target"], groups=train_df["well"]))

# ---------------------------------------------------------------------------
# 6) LightGBM per-seed (3 seeds) + XGB
# ---------------------------------------------------------------------------
LGB_BASE = dict(
    boosting_type="gbdt",
    learning_rate=0.06,
    num_leaves=89,
    min_child_samples=10,
    min_child_weight=0.5,
    n_estimators=5000,
    n_jobs=-1,
    reg_alpha=2.03,
    reg_lambda=87.28,
    subsample=0.645,
    subsample_freq=1,
    colsample_bytree=0.821,
    objective="regression",
    metric="rmse",
    verbose=-1,
)
if USE_GPU:
    LGB_BASE.update(device_type="gpu", gpu_use_dp=False, max_bin=255)


def train_lgb(seed):
    logger.info("LGB seed=%d", seed)
    params = dict(LGB_BASE)
    params["random_state"] = seed
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)
    for fold, (tr, va) in enumerate(splits):
        dtr = lgb.Dataset(train_df.iloc[tr][feature_cols], label=train_df.iloc[tr]["target"])
        dva = lgb.Dataset(train_df.iloc[va][feature_cols], label=train_df.iloc[va]["target"], reference=dtr)
        m = lgb.train(
            params, dtr, valid_sets=[dva],
            num_boost_round=params["n_estimators"],
            callbacks=[lgb.early_stopping(125, verbose=False),
                       lgb.log_evaluation(period=500)],
        )
        oof[va] = m.predict(train_df.iloc[va][feature_cols], num_iteration=m.best_iteration).astype(np.float32)
        rmse = float(np.sqrt(np.mean((oof[va] - train_df.iloc[va]["target"].values) ** 2)))
        logger.info("  fold %d: rmse=%.4f best_iter=%d", fold, rmse, m.best_iteration)
        test_pred += m.predict(test_df[feature_cols], num_iteration=m.best_iteration).astype(np.float32) / N_SPLITS
    overall = float(np.sqrt(np.mean((oof - train_df["target"].values) ** 2)))
    logger.info("LGB seed=%d: OOF rmse=%.4f", seed, overall)
    return oof, test_pred, overall


XGB_BASE = dict(
    objective="reg:squarederror",
    eval_metric="rmse",
    learning_rate=0.06,
    max_depth=8,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=20.0,
    tree_method="hist",
    n_jobs=-1,
)
if USE_GPU:
    XGB_BASE.update(device="cuda")


def train_xgb(seed):
    if not HAS_XGB:
        return None, None, None
    logger.info("XGB seed=%d", seed)
    params = dict(XGB_BASE); params["seed"] = seed
    oof = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)
    for fold, (tr, va) in enumerate(splits):
        dtr = xgb.DMatrix(train_df.iloc[tr][feature_cols].values, label=train_df.iloc[tr]["target"].values)
        dva = xgb.DMatrix(train_df.iloc[va][feature_cols].values, label=train_df.iloc[va]["target"].values)
        m = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dva, "val")], early_stopping_rounds=125, verbose_eval=500)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1)).astype(np.float32)
        rmse = float(np.sqrt(np.mean((oof[va] - train_df.iloc[va]["target"].values) ** 2)))
        logger.info("  fold %d: rmse=%.4f best_iter=%d", fold, rmse, m.best_iteration)
        dte = xgb.DMatrix(test_df[feature_cols].values)
        test_pred += m.predict(dte, iteration_range=(0, m.best_iteration + 1)).astype(np.float32) / N_SPLITS
    overall = float(np.sqrt(np.mean((oof - train_df["target"].values) ** 2)))
    logger.info("XGB seed=%d: OOF rmse=%.4f", seed, overall)
    return oof, test_pred, overall


results = {}
for seed in LGB_SEEDS:
    oof, tp, score = train_lgb(seed)
    results[f"lgb_{seed}"] = {"oof": oof, "test": tp, "rmse": score}

if HAS_XGB:
    oof_xgb, test_xgb, rmse_xgb = train_xgb(42)
    if oof_xgb is not None:
        results["xgb_42"] = {"oof": oof_xgb, "test": test_xgb, "rmse": rmse_xgb}

# ---------------------------------------------------------------------------
# 7) Ensemble: simple average vs ridge stack
# ---------------------------------------------------------------------------
oof_avg = np.mean([r["oof"] for r in results.values()], axis=0)
test_avg = np.mean([r["test"] for r in results.values()], axis=0)
rmse_avg = float(np.sqrt(np.mean((oof_avg - train_df["target"].values) ** 2)))
logger.info("simple avg OOF rmse = %.4f", rmse_avg)

stack_X = np.column_stack([r["oof"] for r in results.values()])
ridge = Ridge(alpha=1.0, fit_intercept=False, positive=True)
ridge.fit(stack_X, train_df["target"].values)
stack_oof = ridge.predict(stack_X)
rmse_stack = float(np.sqrt(np.mean((stack_oof - train_df["target"].values) ** 2)))
weights = ridge.coef_ / max(ridge.coef_.sum(), 1e-9)
logger.info("ridge OOF rmse = %.4f weights=%s", rmse_stack,
            {k: float(round(w, 3)) for k, w in zip(results.keys(), weights)})
stack_test = ridge.predict(np.column_stack([r["test"] for r in results.values()]))

if rmse_avg <= rmse_stack:
    final_test = test_avg
    final_oof = oof_avg
    final_rmse = rmse_avg
    logger.info("Final: simple average")
else:
    final_test = stack_test
    final_oof = stack_oof
    final_rmse = rmse_stack
    logger.info("Final: ridge stack")
logger.info("Final OOF rmse: %.4f", final_rmse)

# ---------------------------------------------------------------------------
# 8) Reconstruct absolute TVT and apply EWM(span=4) post-smoothing
# ---------------------------------------------------------------------------
test_anchor = test_df["last_known_tvt"].to_numpy(dtype=np.float64)
test_tvt = test_anchor + final_test.astype(np.float64)

submission = pd.DataFrame({
    "well": test_df["well"].values,
    "row_idx": test_df["row_idx"].astype(np.int32).values,
    "id": test_df["prediction_id"].values,
    "tvt": test_tvt,
}).sort_values(["well", "row_idx"]).reset_index(drop=True)


def _apply_ewm(group):
    g = group.copy()
    g["tvt"] = g["tvt"].ewm(span=EWM_SPAN, adjust=False).mean()
    return g


pre_ewm_tvt = submission["tvt"].copy()
submission = submission.groupby("well", group_keys=False).apply(_apply_ewm)
ewm_change = float(np.mean(np.abs(submission["tvt"].values - pre_ewm_tvt.values)))
logger.info("EWM(span=%.1f) avg |delta| = %.3f ft", EWM_SPAN, ewm_change)

submission_out = submission[["id", "tvt"]].copy()
if submission_out["tvt"].isna().any():
    n_bad = int(submission_out["tvt"].isna().sum())
    logger.warning("NaN tvt in %d rows; backfilling with last_known_tvt", n_bad)
    bad = submission_out["tvt"].isna()
    submission_out.loc[bad, "tvt"] = test_anchor[bad.to_numpy()]

if not np.isfinite(submission_out["tvt"]).all():
    n_bad = int((~np.isfinite(submission_out["tvt"])).sum())
    median_tvt = float(np.median(test_anchor[np.isfinite(test_anchor)]))
    logger.warning("Non-finite tvt in %d rows; replacing with median=%.2f", n_bad, median_tvt)
    bad = ~np.isfinite(submission_out["tvt"])
    submission_out.loc[bad, "tvt"] = median_tvt

submission_out.to_csv(OUTPUT, index=False)
oof_df = pd.DataFrame({
    "prediction_id": train_df["prediction_id"],
    "well": train_df["well"],
    "row_idx": train_df["row_idx"].astype(np.int32),
    "target": train_df["target"].values,
    "oof_pred": final_oof.astype(np.float64),
    "last_known_tvt": train_df["last_known_tvt"].astype(np.float64),
})
oof_df.to_csv(OOF_OUT, index=False)

logger.info("Wrote %s (%d rows) and %s", OUTPUT, len(submission_out), OOF_OUT)
print(f"Submission: {len(submission_out)} rows, {submission_out['id'].nunique()} unique ids")
print("TVT stats:")
print(submission_out["tvt"].describe())
print("Head:")
print(submission_out.head(10))
print("Tail:")
print(submission_out.tail(10))
print(f"Final OOF rmse: {final_rmse:.4f}")
